# Using `apl` magics
> An introduction to `aplnb`

`aplnb` adds `%apl` and `%%apl` magics to Jupyter and IPython, which execute code in Dyalog APL. It talks to the interpreter over the [RIDE protocol](https://github.com/Dyalog/ride/blob/master/docs/protocol.md), the same protocol Dyalog's own IDE uses, so nothing needs to be loaded into your workspace, output looks exactly like a Dyalog session, and errors are detected reliably. See [`core`](https://answerdotai.github.io/aplnb/core.html) for a step by step walkthrough of the protocol and of how aplnb is built on it.


## Installation

First, install [Dyalog APL](https://www.dyalog.com/). Dyalog provides a basic license for free. aplnb is developed against Dyalog 20.0; 18.2 and later may work but are untested. Once Dyalog is installed, install aplnb with:

```
pip install aplnb
```

Once that's complete, you can install the magics to all IPython and Jupyter sessions automatically by running in your terminal:

```
aplnb_install
```

The Dyalog interpreter itself only starts the first time you run an `apl` magic, so having aplnb installed everywhere costs nothing when you don't use it.


## Usage

After first running an `apl` magic in a notebook, the [APL language bar](https://abrudz.github.io/lb/apl) by Adám Brudzewsky is automatically added to the current page. (aplnb bundles a modified copy of Adám's lb.js; the file header lists the changes. The most visible ones: type a backtick twice in a row to enter triple backticks, get a `⋄` glyph with backtick-q, and use the `▲`/`▼` button beside the close button to choose whether the bar pushes the page down or overlays it, remembered per site.)

The cell magic (`%%apl`) runs APL code and prints the session's own output, so results look exactly as they do in Dyalog:


In [ ]:
#|hide
%load_ext aplnb

In [ ]:
%%apl
m←3 3⍴⍳9
m×10


10 20 30
40 50 60
70 80 90


<IPython.core.display.Javascript object>

Assignments are shy, just like in the Dyalog session, so the `m←` line above printed nothing. The line magic (`%apl`) instead evaluates one expression and returns it as a Python value:

In [ ]:
%apl 3×⍳4

[3, 6, 9, 12]

In [ ]:
%apl ⎕A

'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

Because the line magic returns a value, you can store it in a Python variable. Scalars come back as numbers or strings, vectors as lists, and higher-rank arrays as nested lists:


In [ ]:
z = %apl m
z


[[1, 2, 3], [4, 5, 6], [7, 8, 9]]

To suppress a cell's output, end the last line with a `;`:


In [ ]:
%%apl
m×10;


`⎕←` displays a value explicitly, which is how you show something that would otherwise be shy:


In [ ]:
%%apl
v←2×⍳5
⎕←v


2 4 6 8 10


To use numpy, just pass the result of `%apl` into `np.array`:

In [ ]:
import numpy as np

In [ ]:
a = %apl m
np.array(a)


array([[1, 2, 3],
       [4, 5, 6],
       [7, 8, 9]])

### Example algorithms

The fibonacci sequence:

In [ ]:
%apl {⍵,+/¯2↑⍵}⍣15⊢1 1

[1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597]

Explanation:

1. `1 1`: Initial seed (first two Fibonacci numbers)
2. `{⍵,+/¯2↑⍵}`: Function that appends the sum of the last two elements
3. `⍣15`: Apply the function 15 times
4. `⊢`: Identity function, passes the initial argument (1 1) to the iteration

Prime number sieve:

In [ ]:
%%apl
primes ← {⍵×2=+⌿0=⍵∘.|⍵}⍳
(primes 50)~0

2 3 5 7 11 13 17 19 23 29 31 37 41 43 47


Explanation:

1. `⍳50` generates integers 1 to 50
2. `⍵∘.|⍵` creates a 50x50 matrix of divisibility (1 if divisible, 0 if not)
3. `0=` inverts the matrix (1 for non-divisible)
4. `+⌿` sums columns, counting non-divisors for each number
5. `2=` checks if count equals 2 (prime property)
6. `⍵×` multiplies result with original numbers, keeping primes
7. `~0` removes zero from the result

## Using aplnb from Python

The magics are a thin layer over the `Apl` class, which you can use directly in scripts, tests, and other tooling. Calling the session runs code and returns the output exactly as Dyalog formats it (or None if there's no output); APL errors raise `AplError`:

In [ ]:
from aplnb import Apl

apl = Apl()
apl('3 3⍴⍳9')

1 2 3
4 5 6
7 8 9

Square brackets move values between Python and the workspace, in both directions, and take any expression:

In [ ]:
apl['v'] = [3,1,4,1,5]
apl['{⍵[⍋⍵]}v']

[1, 1, 3, 4, 5]

`fn` lifts an APL function into a Python callable (one argument applies it monadically, two dyadically):

In [ ]:
mean = apl.fn('{(+/⍵)÷≢⍵}')
mean([1,2,3,4])

2.5

Sessions shut themselves down at process exit; use `close`, or a `with Apl() as apl:` block, to do it sooner:

In [ ]:
apl.close()

## Limitations

- Keyboard input through `⎕` or `⍞` can't work in a notebook, so it raises an error. The session survives.
- A cell that ends inside an unfinished block, such as an unclosed `:If`, wedges the interpreter with no way back ([Dyalog/ride#1401](https://github.com/Dyalog/ride/issues/1401)). aplnb detects this, tells you, and starts a fresh session, but workspace state is lost when it happens.
- `%apl` transfers values with `⎕JSON`, so it's limited to arrays and scalars that JSON can represent, serializing to at most 32767 characters. For bigger data, write a file from APL instead.
- Windows isn't supported yet: aplnb configures Dyalog through environment variables, which is the Unix convention. macOS and Linux are supported.

## Learning APL

To start learning APL, follow the [17 video series](https://forums.fast.ai/t/apl-array-programming/97188) run by Jeremy Howard, and have a look at the [study notes](https://fastai.github.io/apl-study/apl.html). The `]` user commands mentioned there, such as `]Help ≠`, work in aplnb too.
